In [ ]:
## Setup — packages & environment

import sys
import subprocess

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

required = ['pandas', 'numpy', 'scipy', 'matplotlib', 'seaborn', 'scikit-learn', 'openpyxl', 'reportlab']

for pkg in required:
    try:
        __import__(pkg.split('-')[0])
    except Exception:
        install(pkg)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import datetime as dt
from scipy import stats
from sklearn.preprocessing import StandardScaler

RSEED = 2023
np.random.seed(RSEED)
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

In [ ]:
try:
    import ipykernel
    print('ipykernel installed')
except:
    print('ipykernel not installed')

In [ ]:
## 1. Multi-Channel Event Data

# Create synthetic multi-channel event sequences
np.random.seed(RSEED)
channels = ['forum', 'quiz', 'assignment']
n_students = 30
n_events = 100

events_list = []
for student_id in range(n_students):
    # Generate event sequence with mixing patterns
    for event_idx in range(n_events):
        channel = np.random.choice(channels, p=[0.4, 0.35, 0.25])
        timestamp = event_idx + np.random.normal(0, 0.5)
        events_list.append({
            'student_id': f'student_{student_id}',
            'timestamp': timestamp,
            'channel': channel,
            'duration': max(1, np.random.normal(10, 3))
        })

df = pd.DataFrame(events_list).sort_values(['student_id', 'timestamp']).reset_index(drop=True)
print('Multi-channel Event Data:')
print(df.head(10))
print(f'\nTotal events: {len(df)}')
print(f'Channel distribution:')
print(df['channel'].value_counts())

In [ ]:
## 2. Channel Sequences and Alignment

# Extract per-channel sequences for each student
student_channels = {}
for student_id in df['student_id'].unique():
    student_data = df[df['student_id'] == student_id]
    student_channels[student_id] = {}
    for channel in channels:
        channel_events = student_data[student_data['channel'] == channel].sort_values('timestamp')
        student_channels[student_id][channel] = list(range(len(channel_events)))

# Summary statistics per channel
print('Events per Channel and Student:')
for student_id in list(student_channels.keys())[:5]:
    print(f'\n{student_id}:')
    for channel in channels:
        count = len(student_channels[student_id][channel])
        print(f'  {channel}: {count}')

In [ ]:
## 3. Cross-Channel Dependency Analysis

# Compute co-occurrence matrix (which channels appear together)
co_occurrence = pd.DataFrame(0, index=channels, columns=channels)

for student_id in df['student_id'].unique():
    student_data = df[df['student_id'] == student_id].sort_values('timestamp')
    # Look at adjacent events
    for i in range(len(student_data) - 1):
        curr_channel = student_data.iloc[i]['channel']
        next_channel = student_data.iloc[i + 1]['channel']
        co_occurrence.loc[curr_channel, next_channel] += 1

# Normalize
co_occurrence_prob = co_occurrence.div(co_occurrence.sum(axis=1), axis=0).fillna(0)

print('Cross-Channel Transition Probabilities:')
print(co_occurrence_prob.round(3))

In [ ]:
## 4. Visualizations

os.makedirs('figures', exist_ok=True)

# 1. Channel event counts
fig, ax = plt.subplots(figsize=(9, 6))
channel_counts = df['channel'].value_counts()
colors = sns.color_palette('Set2', len(channels))
ax.bar(range(len(channel_counts)), channel_counts.values, color=colors, edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(channel_counts)))
ax.set_xticklabels(channel_counts.index)
ax.set_ylabel('Number of Events', fontsize=12)
ax.set_title('Total Events per Channel', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(channel_counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/01_channel_counts.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 01_channel_counts.png')

# 2. Cross-channel transitions
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(co_occurrence_prob, annot=True, fmt='.3f', cmap='Blues', square=True, ax=ax)
ax.set_title('Cross-Channel Transition Probabilities', fontsize=14, fontweight='bold')
ax.set_xlabel('To Channel', fontsize=12)
ax.set_ylabel('From Channel', fontsize=12)
plt.tight_layout()
plt.savefig('figures/02_cross_channel_transitions.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 02_cross_channel_transitions.png')

# 3. Timeline of events by channel
fig, ax = plt.subplots(figsize=(12, 6))
channel_colors = {'forum': 'red', 'quiz': 'blue', 'assignment': 'green'}
for channel in channels:
    channel_data = df[df['channel'] == channel]
    ax.scatter(channel_data['timestamp'], [channel] * len(channel_data), 
              alpha=0.6, s=50, label=channel, color=channel_colors[channel])
ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Channel', fontsize=12)
ax.set_title('Event Timeline by Channel', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/03_event_timeline.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 03_event_timeline.png')

df.to_csv('multichannel_events.csv', index=False)
print('\nSaved multi-channel event data')

In [ ]:
## 5. PDF Handout Generation

from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak
from reportlab.lib.enums import TA_JUSTIFY

pdf_path = 'Ch13_MultiChannelSequence_Handout.pdf'
doc = SimpleDocTemplate(pdf_path, pagesize=letter, rightMargin=0.75*inch, leftMargin=0.75*inch,
                        topMargin=0.75*inch, bottomMargin=0.75*inch)

styles = getSampleStyleSheet()
styleN = styles['Normal']
styleN.fontSize = 11
styleN.alignment = TA_JUSTIFY

story = []
story.append(Paragraph('<b>Chapter 13: Multi-Channel Sequence Analysis</b>', styles['Heading1']))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>1. Introduction</b>', styles['Heading2']))
intro = (
    'Multi-channel sequence analysis extends simple sequences to include multiple concurrent event streams. '
    'Students interact with courses through different channels (forums, quizzes, assignments). '
    'Understanding cross-channel patterns reveals how different learning modalities interact.'
)
story.append(Paragraph(intro, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>2. Dataset & Channels</b>', styles['Heading2']))
dataset = (
    f'Event sequences from {df["student_id"].nunique()} students across {len(channels)} channels '
    f'({', '.join(channels)}). Total events: {len(df)}.'
)
story.append(Paragraph(dataset, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>3. Methods</b>', styles['Heading2']))
methods = (
    'Events were extracted and timestamped. Cross-channel transition probabilities computed '
    'by examining consecutive events across all channels. Temporal patterns visualized.'
)
story.append(Paragraph(methods, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>4. Results</b>', styles['Heading2']))
story.append(Spacer(1, 6))

try:
    if os.path.exists('figures/01_channel_counts.png'):
        story.append(Image('figures/01_channel_counts.png', width=480, height=300))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 1: Event distribution across channels.', styleN))
        story.append(Spacer(1, 12))
except: pass

try:
    if os.path.exists('figures/02_cross_channel_transitions.png'):
        story.append(Image('figures/02_cross_channel_transitions.png', width=420, height=360))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 2: Cross-channel transition patterns.', styleN))
        story.append(Spacer(1, 12))
except: pass

story.append(PageBreak())
story.append(Paragraph('<b>5. Interpretation</b>', styles['Heading2']))
interp = (
    'Cross-channel analysis reveals students\'  preferred navigation patterns and learning pathways. '
    'High transition probabilities between specific channels indicate integrated learning activities.'
)
story.append(Paragraph(interp, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>6. Limitations</b>', styles['Heading2']))
lim = (
    'Analysis assumes channel types are independent, though students may employ strategic navigation. '
    'Time gaps between channels not explicitly modeled.'
)
story.append(Paragraph(lim, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph(f'Generated on: {dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', styleN))

try:
    doc.build(story)
    print(f'Saved PDF: {pdf_path}')
except Exception as e:
    print(f'PDF generation failed: {e}')